# <u>Step 5 :- </u> **States & The Discount Factor ($\gamma$)**

**From Bandits to MDPs** :- In previous steps, there was only 1 situation (stateless). Now the agent moves across different positions (**States**)! <br>
**Episode & Terminal State** :- A sequence of actions from start until reaching the goal (`done = True`). <br>
**The Discount Factor ($\gamma$)** :- Controls how much the agent values future rewards vs immediate rewards ($0 \le \gamma \le 1$). <br>
**Discounted Return Formula** :-
$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

## 1. Computing Discounted Return ($G$) :-

Future rewards are discounted by $\gamma^{\text{steps}}$:
- **If $\gamma = 0$** : Only immediate reward matters (completely short-sighted / myopic).
- **If $\gamma \to 1$** : Future rewards matter almost as much as immediate rewards (far-sighted).

In [1]:
def compute_return(gamma, rewards):
    total_return = 0
    for steps, r in enumerate(rewards):
        total_return += r * (gamma ** steps)
    return total_return

## 2. Environment (1D State World) :-

- **States** : `[0] -> [1] -> [2] -> [3 (Goal)]`
- **Actions** : `['left', 'right']`
- **Step penalty** : `-1` (encourages the agent to reach the goal quickly)
- **Goal reward** : `+10` at state `3` with `done = True`

In [2]:
actions = ['left', 'right']

def step(state, action):
    if action == 'right':
        next_state = state + 1
    else:
        next_state = state - 1

    # Boundary check: cannot go left past state 0
    if next_state < 0:
        next_state = 0

    if next_state == 3:
        reward = 10
        done = True
    else:
        reward = -1
        done = False

    return next_state, reward, done

## 3. Running an Episode :-

Starting at `state = 0` and repeatedly taking `'right'` until reaching the goal.

In [3]:
gamma = 0.2
done = False
state = 0
rewards = []

print("Starting at State :", state)
while not done:
    state, reward, done = step(state, 'right')
    rewards.append(reward)

    print("State :", state, "| Reward :", reward, "| Done :", done)

print("\nCollected Rewards :", rewards)

total_return = compute_return(gamma, rewards)
print(f"Total Return (gamma={gamma}) :", round(total_return, 4))

Starting at State : 0
State : 1 | Reward : -1 | Done : False
State : 2 | Reward : -1 | Done : False
State : 3 | Reward : 10 | Done : True

Collected Rewards : [-1, -1, 10]
Total Return (gamma=0.2) : -0.8


## 4. Why $\gamma$ Matters: Comparing Discount Factors :-

With $\gamma = 0.2$:
$$G = (-1 \times 0.2^0) + (-1 \times 0.2^1) + (10 \times 0.2^2) = -1 - 0.2 + 0.4 = -0.8$$
Because $\gamma = 0.2$ was too low, the $+10$ goal was discounted to only $+0.4$, so the agent viewed the path as negative!

Let's see what happens to the return when we increase $\gamma$:

In [4]:
gammas = [0.0, 0.2, 0.5, 0.8, 0.9, 0.99]

print("Rewards sequence :", rewards, "\n")
for g in gammas:
    ret = compute_return(g, rewards)
    print(f"gamma = {g:<4} -> Total Return G = {round(ret, 4)}")

Rewards sequence : [-1, -1, 10]

gamma = 0.0  -> Total Return G = -1.0
gamma = 0.2  -> Total Return G = -0.8
gamma = 0.5  -> Total Return G = 1.0
gamma = 0.8  -> Total Return G = 4.6
gamma = 0.9  -> Total Return G = 6.2
gamma = 0.99 -> Total Return G = 7.811


# **challenge** : <u>Return from Every State ($G_t$)</u>

In RL, an agent needs to know how good being in state $s$ is.
That is determined by the return $G_t$ from that point onward to the goal (using $\gamma = 0.9$):

In [5]:
gamma = 0.9

for t in range(len(rewards)):
    remaining_rewards = rewards[t:]
    return_from_t = compute_return(gamma, remaining_rewards)
    print(f"State {t} -> Goal : Remaining Rewards {remaining_rewards} | Return G_{t} = {round(return_from_t, 3)}")

State 0 -> Goal : Remaining Rewards [-1, -1, 10] | Return G_0 = 6.2
State 1 -> Goal : Remaining Rewards [-1, 10] | Return G_1 = 8.0
State 2 -> Goal : Remaining Rewards [10] | Return G_2 = 10.0
